# Liu-like sepsis pipeline\n\nThis notebook runs the full modular pipeline for:\n- GLM\n- XGBoost (if available)\n- GRU\n\nIt also computes patient-time early warning metrics and saves outputs.

In [ ]:
from pathlib import Path\nimport json\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nfrom IPython.display import display\nfrom tqdm.notebook import tqdm\n\nfrom config import ExperimentConfig\nfrom data import DataModule\nfrom evaluation import EvaluationModule\nfrom models.glm_model import LiuLikeGLM\nfrom models.xgb_model import LiuLikeXGBoost\nfrom models.gru_model import LiuLikeGRU\n\nwarnings.filterwarnings("ignore")\n\nprint("cwd:", Path.cwd())\nprint("files:", sorted([p.name for p in Path(".").iterdir()]))

In [ ]:
cfg = ExperimentConfig()\ncfg.data_dir = Path("./data")\ncfg.output_dir = Path("./results_notebook")\ncfg.ensure_output_dir()\n\ndm = DataModule(cfg)\nev = EvaluationModule()\n\ndm.set_global_seed(cfg.seed)\n\nprint("DATA_DIR:", cfg.data_dir.resolve() if cfg.data_dir.exists() else cfg.data_dir)\nprint("OUTPUT_DIR:", cfg.output_dir.resolve())\nprint("seed:", cfg.seed)

## Load and prepare data

In [ ]:
top_bar = tqdm(total=4, desc="Notebook pipeline", position=0, leave=True)\n\nprint("Loading all patients...")\ndf, summaries = dm.load_all_patients()\ntop_bar.update(1)\n\nprint("Splitting patients...")\ntrain_ids, test_ids = dm.split_patients(summaries, test_size=cfg.test_size, seed=cfg.seed)\ntop_bar.update(1)\n\nprint("Engineering features...")\ndf_feat = dm.add_engineered_features(df)\ntop_bar.update(1)\n\nprint("Building static dataset...")\nstatic_ds = dm.build_static_dataset(df_feat, train_ids, test_ids)\ntop_bar.update(1)\n\ntop_bar.close()\n\nsummary_preview = pd.DataFrame([s.__dict__ for s in summaries])\ndisplay(summary_preview.head())\n\nprint("Full data shape:", df.shape)\nprint("Feature-engineered shape:", df_feat.shape)\nprint("Train patients:", len(train_ids))\nprint("Test patients:", len(test_ids))\nprint("Number of modeling features:", len(static_ds.feature_cols))\nprint("First 10 features:", static_ds.feature_cols[:10])\nprint("Static X_train:", static_ds.X_train.shape, "positives:", int(np.sum(static_ds.y_train)))\nprint("Static X_test:", static_ds.X_test.shape, "positives:", int(np.sum(static_ds.y_test)))

## GLM

In [ ]:
glm_bar = tqdm(total=3, desc="GLM", leave=True)\n\nglm = LiuLikeGLM(random_state=cfg.seed)\nglm.fit(static_ds.X_train, static_ds.y_train)\nglm_bar.update(1)\n\nglm_probs = glm.predict_proba(static_ds.X_test)\nglm_metrics = ev.evaluate_binary_probabilities(static_ds.y_test, glm_probs)\nglm_bar.update(1)\n\nglm_scores = ev.build_full_patient_time_scores_static(\n    glm,\n    df_feat,\n    test_ids,\n    static_ds.feature_cols\n)\n\nglm_ewt = ev.compute_early_warning_metrics(\n    glm_scores,\n    glm_metrics.optimal_threshold,\n    cfg.label_lead_hours\n)\nglm_bar.update(1)\nglm_bar.close()\n\nglm_scores.to_csv(cfg.output_dir / "glm_test_time_scores.csv", index=False)\n\nprint("GLM row-level metrics")\ndisplay(pd.DataFrame([glm_metrics.__dict__]))\nprint("GLM patient-time metrics")\ndisplay(pd.DataFrame([glm_ewt]))

## XGBoost

In [ ]:
xgb = None\nxgb_metrics = None\nxgb_ewt = None\nxgb_scores = None\n\ntry:\n    xgb_bar = tqdm(total=3, desc="XGBoost", leave=True)\n    xgb = LiuLikeXGBoost(random_state=cfg.seed)\n    xgb.fit(static_ds.X_train, static_ds.y_train)\n    xgb_bar.update(1)\n\n    xgb_probs = xgb.predict_proba(static_ds.X_test)\n    xgb_metrics = ev.evaluate_binary_probabilities(static_ds.y_test, xgb_probs)\n    xgb_bar.update(1)\n\n    xgb_scores = ev.build_full_patient_time_scores_static(\n        xgb,\n        df_feat,\n        test_ids,\n        static_ds.feature_cols\n    )\n\n    xgb_ewt = ev.compute_early_warning_metrics(\n        xgb_scores,\n        xgb_metrics.optimal_threshold,\n        cfg.label_lead_hours\n    )\n    xgb_bar.update(1)\n    xgb_bar.close()\n\n    xgb_scores.to_csv(cfg.output_dir / "xgb_test_time_scores.csv", index=False)\n\n    print("XGBoost row-level metrics")\n    display(pd.DataFrame([xgb_metrics.__dict__]))\n    print("XGBoost patient-time metrics")\n    display(pd.DataFrame([xgb_ewt]))\n\nexcept Exception as e:\n    print("XGBoost skipped:", repr(e))

## GRU

In [ ]:
print("Building GRU sequences...")\nseq_bar = tqdm(total=2, desc="GRU sequence prep", leave=True)\n\ntrain_seq_X, train_seq_y = dm.build_patient_sequences(\n    df_feat,\n    train_ids,\n    static_ds.feature_cols\n)\nseq_bar.update(1)\n\ntest_seq_X, test_seq_y = dm.build_patient_sequences(\n    df_feat,\n    test_ids,\n    static_ds.feature_cols\n)\nseq_bar.update(1)\nseq_bar.close()\n\nprint("Train sequences:", train_seq_X.shape, "positives:", int(np.sum(train_seq_y)))\nprint("Test sequences:", test_seq_X.shape, "positives:", int(np.sum(test_seq_y)))

In [ ]:
gru_bar = tqdm(total=3, desc="GRU", leave=True)\n\ngru = LiuLikeGRU(\n    random_state=cfg.seed,\n    hidden_size=cfg.rnn_hidden_size,\n    num_layers=cfg.rnn_num_layers,\n    dropout=cfg.rnn_dropout,\n    epochs=cfg.epochs,\n    batch_size=cfg.batch_size,\n    learning_rate=cfg.learning_rate,\n)\n\ngru.fit(train_seq_X, train_seq_y)\ngru_bar.update(1)\n\ngru_probs = gru.predict_proba(test_seq_X)\ngru_metrics = ev.evaluate_binary_probabilities(test_seq_y, gru_probs)\ngru_bar.update(1)\n\ngru_scores = ev.build_full_patient_time_scores_rnn(\n    gru,\n    df_feat,\n    test_ids,\n    static_ds.feature_cols,\n    seq_len=cfg.seq_len\n)\n\ngru_ewt = ev.compute_early_warning_metrics(\n    gru_scores,\n    gru_metrics.optimal_threshold,\n    cfg.label_lead_hours\n)\ngru_bar.update(1)\ngru_bar.close()\n\ngru_scores.to_csv(cfg.output_dir / "gru_test_time_scores.csv", index=False)\n\nprint("GRU row-level metrics")\ndisplay(pd.DataFrame([gru_metrics.__dict__]))\nprint("GRU patient-time metrics")\ndisplay(pd.DataFrame([gru_ewt]))

## Save metrics and summary

In [ ]:
results = {\n    "glm_row": glm_metrics.__dict__,\n    "glm_patient_time": glm_ewt,\n    "gru_row": gru_metrics.__dict__,\n    "gru_patient_time": gru_ewt,\n}\n\nif xgb_metrics is not None and xgb_ewt is not None:\n    results["xgb_row"] = xgb_metrics.__dict__\n    results["xgb_patient_time"] = xgb_ewt\n\nwith open(cfg.output_dir / "metrics.json", "w") as f:\n    json.dump(results, f, indent=2)\n\nwith open(cfg.output_dir / "patient_split.json", "w") as f:\n    json.dump({\n        "train_patient_ids": train_ids,\n        "test_patient_ids": test_ids\n    }, f, indent=2)\n\nsummary_rows = [\n    {\n        "model": "GLM",\n        "AUC": results["glm_row"]["auc"],\n        "Sensitivity": results["glm_row"]["sensitivity_at_optimal_threshold"],\n        "Specificity": results["glm_row"]["specificity_at_optimal_threshold"],\n        "Median Early Warning (hrs)": results["glm_patient_time"]["median_early_warning_time_hours"],\n    },\n    {\n        "model": "GRU",\n        "AUC": results["gru_row"]["auc"],\n        "Sensitivity": results["gru_row"]["sensitivity_at_optimal_threshold"],\n        "Specificity": results["gru_row"]["specificity_at_optimal_threshold"],\n        "Median Early Warning (hrs)": results["gru_patient_time"]["median_early_warning_time_hours"],\n    },\n]\n\nif "xgb_row" in results:\n    summary_rows.insert(1, {\n        "model": "XGBoost",\n        "AUC": results["xgb_row"]["auc"],\n        "Sensitivity": results["xgb_row"]["sensitivity_at_optimal_threshold"],\n        "Specificity": results["xgb_row"]["specificity_at_optimal_threshold"],\n        "Median Early Warning (hrs)": results["xgb_patient_time"]["median_early_warning_time_hours"],\n    })\n\nsummary_df = pd.DataFrame(summary_rows)\n\nprint("Saved outputs to:", cfg.output_dir.resolve())\ndisplay(pd.DataFrame(results).T)\ndisplay(summary_df)